# Distress Gesture Detection — Training Pipeline

**Before running any cell, complete these 3 setup steps:**

### 1. Enable GPU
Top-right → Settings (⚙) → Accelerator → **GPU P100** → Save

### 2. Add Datasets (click + Add Data, search each name)
| Dataset | Search for |
|---|---|
| UR Fall Detection | `ur-fall-detection-dataset` by shahliza27 |
| Le2i Fall | `falldataset-imvia` by tuyenldvn |
| NTU RGB+D 60 Skeleton | `skeleton-data-of-ntu-rgbd-60-dataset` by hungkhoi |
| CCTV Weapon | `cctv-weapon-dataset` by simuletic |
| CCTV ATM Robbery | `cctv-atm-robbery-detection-dataset-gun-and-knife` by simuletic |
| Surveillance Knife | `surveillance-vlm-weapon-and-knife-detection-dataset` by simuletic |

### 3. Turn on Internet
Settings (⚙) → Internet → **On**

---
**Then run all cells top to bottom. Each step prints its own status.**

## Cell 1 — Verify GPU + Clone Repo

In [ ]:
import subprocess, os, sys
from pathlib import Path

# Check GPU
gpu = subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print(f'GPU: {gpu}')

# Check datasets
print('\nAttached datasets:')
for d in sorted(Path('/kaggle/input').iterdir()):
    print(f'  /kaggle/input/{d.name}')

# Clone repo
REPO_URL = 'https://github.com/shrishri12062000/distress.git'
REPO_DIR = '/kaggle/working/distress-gesture-detection'

if Path(REPO_DIR).exists():
    print('\nRepo already cloned — pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('\nCloning repo...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        print('ERROR: Clone failed. Check internet is ON in Settings.')

sys.path.insert(0, REPO_DIR)
print(f'\nRepo ready at: {REPO_DIR}')

## Cell 2 — Install Dependencies

In [ ]:
print('Installing dependencies (takes ~2 min)...')
os.system(f'pip install -q -r {REPO_DIR}/requirements_kaggle.txt')
os.system('pip install -q mediapipe datasets huggingface-hub')

# Verify key imports
import torch, cv2, mediapipe, ultralytics, onnx, onnxruntime
print(f'torch:        {torch.__version__}  (CUDA: {torch.cuda.is_available()})')
print(f'opencv:       {cv2.__version__}')
print(f'mediapipe:    {mediapipe.__version__}')
print(f'ultralytics:  {ultralytics.__version__}')
print(f'onnx:         {onnx.__version__}')
print('\nAll dependencies ready.')

## Cell 3 — Data Preparation
Extracts MediaPipe skeletons from all video datasets and merges knife datasets.

⏱ **This step takes 30–90 minutes** (skeleton extraction is CPU-bound).
Results are cached so re-running is instant.

In [ ]:
os.chdir(REPO_DIR)
%run kaggle/prepare_data.py

## Cell 4 — Validate Data
Checks everything before training starts.

In [ ]:
%run kaggle/validate_data.py

## Cell 5 — Train ST-GCN
Trains the distress gesture classifier.

⏱ **~2–4 hours on P100 GPU.** Best checkpoint auto-saved.

In [ ]:
%run kaggle/train_stgcn.py

## Cell 6 — Train YOLOv8n Knife Detector
⏱ **~30–60 minutes on P100 GPU.**

In [ ]:
%run kaggle/train_yolo.py

## Cell 7 — Export Models to ONNX
Exports both models for CPU inference on your laptop.

In [ ]:
%run kaggle/export_models.py

## Cell 8 — Verify ONNX Models

In [ ]:
import numpy as np
import onnxruntime as ort
from pathlib import Path

MODELS = Path('/kaggle/working')

# Test ST-GCN
stgcn_path = MODELS / 'stgcn.onnx'
if stgcn_path.exists():
    sess  = ort.InferenceSession(str(stgcn_path), providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 30, 17), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})[0][0]
    probs = np.exp(out) / np.exp(out).sum()
    print('ST-GCN output (dummy input):')
    for name, p in zip(['normal','help_signal','collapse_falling','fall_down'], probs):
        print(f'  {name:<20}: {p:.4f}')
    print('  ✓ ST-GCN ONNX working')
else:
    print('  ST-GCN ONNX not found')

# Test YOLOv8
yolo_path = MODELS / 'yolo_knife.onnx'
if yolo_path.exists():
    sess  = ort.InferenceSession(str(yolo_path), providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})
    print(f'\nYOLOv8 output shape: {out[0].shape}')
    print('  ✓ YOLOv8 ONNX working')
else:
    print('  YOLOv8 ONNX not found')

print('\n' + '='*50)
print('  DOWNLOAD YOUR MODELS:')
print('  Right panel → Output tab → Download:')
print('    stgcn.onnx')
print('    yolo_knife.onnx')
print('  Place in: distress-gesture-detection/models/')
print('='*50)